# Workspaces v2 — endpoint inspection

Walks every endpoint on the `/api/workspaces` tree (plus `/api/module_registry`) and prints both the parsed model and the raw JSON response so you can eyeball server payloads directly.

**Endpoints exercised**

| Resource | Method | Path |
|---|---|---|
| Health | GET | `/health` |
| Module registry | GET | `/module_registry/modules` |
| Module registry | GET | `/module_registry/modules/:id` |
| Workspaces | POST | `/workspaces` |
| Workspaces | GET | `/workspaces` |
| Workspaces | GET | `/workspaces/:id` |
| Workspaces | PUT | `/workspaces/:id` |
| Workspaces | DELETE | `/workspaces/:id` |
| Tabs | POST | `/workspaces/:wid/tabs` |
| Tabs | GET | `/workspaces/:wid/tabs` |
| Tabs | GET | `/workspaces/:wid/tabs/:id` |
| Tabs | PUT | `/workspaces/:wid/tabs/:id` |
| Tabs | DELETE | `/workspaces/:wid/tabs/:id` |
| Modules | POST | `/workspaces/:wid/tabs/:tid/modules` |
| Modules | GET | `/workspaces/:wid/tabs/:tid/modules` |
| Modules | GET | `/workspaces/:wid/tabs/:tid/modules/:id` |
| Modules | PUT | `/workspaces/:wid/tabs/:tid/modules/:id` |
| Modules | DELETE | `/workspaces/:wid/tabs/:tid/modules/:id` |

**Pre-reqs**

1. `.env` in the project root with `MD_AUTH_TOKEN` (and optional `MD_API_BASE_URL`).
2. The `workspaces_v2_api_enabled` Flipper flag enabled for your user — the entire tree returns 404 otherwise.
3. `pip install -e ".[dev,notebook]"` from the project root.

All cells are idempotent, but the **Cleanup** cell at the bottom must be run last to delete what this notebook created.

## Setup

In [ ]:
import json
import time
from pprint import pp

from md_python import MDClient

client = MDClient(version="v2")
print(f"base_url = {client.base_url}")


def raw(endpoint, method="GET", **kwargs):
    """Hit the API directly so we can see the unwrapped JSON envelope."""
    response = client._make_request(method=method, endpoint=endpoint, **kwargs)
    print(f"{method} {endpoint} -> {response.status_code}")
    if response.text:
        try:
            body = response.json()
            print(json.dumps(body, indent=2)[:4000])
            return body
        except ValueError:
            print(response.text[:1000])
    return None


# IDs created during this notebook — used by the cleanup cell at the bottom.
created = {"workspace_id": None, "tab_id": None, "module_id": None}

## 1. Health — sanity check that auth & base URL are correct

In [ ]:
client.health.check()

In [ ]:
_ = raw("/health")

## 2. Module registry

Discovery endpoint — tells you which `item_id`s are available to you and what `settings` keys each one accepts. The server validates `TabModule.settings` against this; pass a key not declared here and you'll get a 400.

In [ ]:
modules = client.module_registry.list()
print(f"available modules: {len(modules)}\n")

# group → ids, for quick orientation
from collections import defaultdict

by_group = defaultdict(list)
for m in modules:
    by_group[m.group].append(m.id)
for group, ids in sorted(by_group.items()):
    print(f"  {group}: {ids}")

In [ ]:
# Raw payload for the list endpoint — note the {"data": [...]} envelope.
_ = raw("/module_registry/modules")

In [ ]:
# Inspect a single registered module — `heading` is the simplest one and what
# we'll place on a tab below.
heading = client.module_registry.get("heading")
print(f"heading.setting_keys() = {heading.setting_keys()}")
print()
pp(heading.input_settings)

In [ ]:
_ = raw("/module_registry/modules/heading")

## 3. Workspaces — top-level container

In [ ]:
# CREATE
ws_name = f"inspect-notebook-{int(time.time())}"
ws = client.workspaces.create(
    name=ws_name,
    description="Created by inspect_workspaces_v2.ipynb — safe to delete.",
)
created["workspace_id"] = str(ws.id)
print("parsed:")
pp(ws)

# Same call, raw — see the JSON shape returned by `present_workspace`.
print("\nraw POST response:")
_ = raw(
    "/workspaces",
    method="POST",
    json={"name": f"{ws_name}-raw", "description": "raw view"},
    headers={"Content-Type": "application/json"},
)

In [ ]:
# Clean up the secondary workspace we made just to inspect the raw create.
# (Pull the latest list, find ones tagged 'raw view', delete them.)
for w in client.workspaces.list_all():
    if w.description == "raw view":
        client.workspaces.delete(str(w.id))
        print(f"deleted secondary workspace {w.id} ({w.name!r})")

In [ ]:
# LIST (paginated, 50 per page)
page = client.workspaces.list(page=1)
print(f"page 1: {len(page['data'])} items")
pp(page["pagination"])

# raw: notice the {"data": [...], "pagination": {...}} envelope
print("\nraw:")
_ = raw("/workspaces", params={"page": 1})

In [ ]:
# SHOW
wid = created["workspace_id"]
fetched = client.workspaces.get(wid)
pp(fetched)

print("\nraw:")
_ = raw(f"/workspaces/{wid}")

In [ ]:
# UPDATE — partial: only the fields you pass are sent
renamed = client.workspaces.update(wid, description="renamed by notebook")
pp(renamed)

print("\nraw:")
_ = raw(
    f"/workspaces/{wid}",
    method="PUT",
    json={"description": "renamed via raw call"},
    headers={"Content-Type": "application/json"},
)

## 4. Tabs — `tab_index` is auto-assigned, `layout` starts as `{modules: []}`

In [ ]:
# CREATE
tab = client.workspaces.tabs.create(wid, name="Inspection Tab")
created["tab_id"] = str(tab.id)
pp(tab)

print("\nraw:")
_ = raw(
    f"/workspaces/{wid}/tabs",
    method="POST",
    json={"name": "raw inspection tab", "settings": {"reportMode": False}},
    headers={"Content-Type": "application/json"},
)

In [ ]:
# Drop the secondary 'raw' tab — keep only the one we'll attach modules to.
for t in client.workspaces.tabs.list_all(wid):
    if t.name == "raw inspection tab":
        client.workspaces.tabs.delete(wid, str(t.id))
        print(f"deleted secondary tab {t.id}")

In [ ]:
# LIST + SHOW
tabs_page = client.workspaces.tabs.list(wid)
print(f"{len(tabs_page['data'])} tab(s) in this workspace")
for t in tabs_page["data"]:
    print(f"  index={t.tab_index} locked={t.locked} {t.name!r} ({t.id})")

tid = created["tab_id"]
print("\nraw single-tab fetch:")
_ = raw(f"/workspaces/{wid}/tabs/{tid}")

In [ ]:
# UPDATE — partial. The PUT body is one of {name, layout, settings}; locked
# tabs reject this with a 403/400.
renamed_tab = client.workspaces.tabs.update(
    wid,
    tid,
    name="Inspection Tab (renamed)",
    settings={"reportMode": True},
)
pp(renamed_tab)

## 5. Modules — placed on a react-grid-layout grid (`x, y, width, height`)

**Note on shape mismatch.** The API surface uses `height`/`width` on POST/PUT, but the persisted JSON inside `tab.layout.modules` (and what each endpoint returns) uses the react-grid-layout shorthand `h`/`w` (and mirrors `id` to `i`). The `TabModule` model normalises both shapes back to `height`/`width`.

**Note on defaults.** The API does **not** merge registry-declared defaults server-side, so a partial `settings` payload persists as-is and the rendered widget in the app then complains it's "missing" keys (e.g. _"Please provide Size / Horizontal Position / Vertical Position"_). Use `client.workspaces.modules.create_with_defaults(...)` to bake every registry default into the payload before it goes over the wire — explicit user values override defaults, missing required keys without a registry default fail fast client-side.

In [ ]:
# CREATE — heading at the top of the tab. The heading module declares
# input_settings keys: text, size, horizontalPosition, verticalPosition.
#
# Using create_with_defaults() so every registry-declared default ends up in
# the persisted settings hash. Without it, the rendered widget would say
# "Please provide Size / Horizontal Position / Vertical Position".
mod = client.workspaces.modules.create_with_defaults(
    workspace_id=wid,
    tab_id=tid,
    item_id="heading",
    x=0,
    y=0,
    width=12,
    height=1,
    settings={"text": "Hello from the inspection notebook"},
    registered_module=heading,  # reuse the one we already fetched above
)
created["module_id"] = str(mod.id)
print("parsed:")
pp(mod)
print(f"\npersisted settings: {mod.settings}")

# A second module via the raw POST — note the partial settings hash here
# *intentionally* skips registry defaults, so this widget will look broken
# in the app. Useful as a visual demonstration of the bug.
print("\nraw POST (intentionally partial — to demonstrate the missing-defaults bug):")
_ = raw(
    f"/workspaces/{wid}/tabs/{tid}/modules",
    method="POST",
    json={
        "item_id": "heading",
        "x": 0,
        "y": 1,
        "width": 12,
        "height": 1,
        "settings": {"text": "raw second heading (partial settings — broken render)"},
    },
    headers={"Content-Type": "application/json"},
)

In [ ]:
# LIST — note this endpoint is NOT paginated (modules per tab are typically
# small) and returns {"data": [...]}.
for m in client.workspaces.modules.list(wid, tid):
    print(
        f"  {m.id} item_id={m.item_id} ({m.width}x{m.height} @ {m.x},{m.y}) "
        f"settings={m.settings!r}"
    )

print("\nraw:")
_ = raw(f"/workspaces/{wid}/tabs/{tid}/modules")

In [ ]:
# SHOW one module
got = client.workspaces.modules.get(wid, tid, created["module_id"])
pp(got)

print("\nraw:")
_ = raw(f"/workspaces/{wid}/tabs/{tid}/modules/{created['module_id']}")

In [ ]:
# UPDATE — move + resize + change settings.
#
# Two things to know on PUT:
# 1. !! Server-side bug workaround !!  The PUT endpoint reads
#    existing['item_id'] from the persisted hash, but persistence stores the
#    camelCase key 'itemId'. A partial update without item_id fails with
#    {"errors": ["item_id can't be blank"]}. Always re-send item_id on PUT
#    until the workflow repo's update.rb is fixed.
# 2. PUT replaces `settings` wholesale — there's no per-key merge. So if you
#    only want to change `text`, you still need to re-send size /
#    horizontalPosition / verticalPosition or the widget will go back to
#    the broken "Please provide ..." render. We rebuild the full settings
#    hash from the registry defaults + our updated text below.
new_settings = {**heading.defaults(), "text": "moved + resized"}

moved = client.workspaces.modules.update(
    wid,
    tid,
    created["module_id"],
    item_id=mod.item_id,  # workaround — see comment above
    x=2,
    y=3,
    width=8,
    height=2,
    settings=new_settings,
)
pp(moved)

## 6. Inspect persisted layout

The tab `present_tab` payload doesn't include `layout`, but you can still see what's stored under the hood by reading the layout off `tab.tab_modules` server-side. We can re-derive the same view client-side via the modules list.

In [ ]:
# Show the full tab payload after our edits.
_ = raw(f"/workspaces/{wid}/tabs/{tid}")

## 7. Cleanup — DELETE everything this notebook created

Run this last. Idempotent — safe to re-run if a step earlier in the notebook errored.

In [ ]:
wid = created["workspace_id"]
tid = created["tab_id"]
mid = created["module_id"]

if wid and tid and mid:
    try:
        client.workspaces.modules.delete(wid, tid, mid)
        print(f"deleted module {mid}")
    except Exception as e:
        print(f"module delete: {e}")

if wid and tid:
    try:
        client.workspaces.tabs.delete(wid, tid)
        print(f"deleted tab {tid}")
    except Exception as e:
        print(f"tab delete: {e}")

if wid:
    try:
        client.workspaces.delete(wid)
        print(f"deleted workspace {wid}")
    except Exception as e:
        print(f"workspace delete: {e}")